# Week 4 Exercise – Python to C Port

Convert Python code to high-performance C using frontier and open-source coding models. Same pattern as the week 4 lab: system info for toolchain, C-only output, compile and run. Gradio UI with model selection, Run Python / Run C.

Run the notebook from the repo root (`llm_engineering`) so `week4/system_info` can be imported.

In [34]:
import os
import sys
import io
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess

load_dotenv(override=True)

cwd = Path.cwd()
if "week4" in str(cwd):
    week4_root = Path(str(cwd).split("week4")[0]) / "week4"
else:
    week4_root = cwd / "week4"
sys.path.insert(0, str(week4_root))

from system_info import retrieve_system_info

In [35]:
openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

openai_client = OpenAI(api_key=openai_api_key, base_url=openrouter_url) if openai_api_key else None
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url) if openrouter_api_key else None
ollama = OpenAI(base_url=ollama_url, api_key="ollama")

MODELS = [
    "openai/gpt-4o",
    "openai/gpt-4o-mini",
    "anthropic/claude-3.5-sonnet",
    "google/gemini-2.0-flash",
    "qwen/qwen-2.5-coder-32b-instruct",
    "deepseek/deepseek-coder-v2",
    "codellama/CodeLlama-34b-Instruct",
]

def get_client(model: str):
    if model.startswith("openai/") and openai_client:
        return openai_client
    if "codellama" in model or "qwen" in model or "deepseek" in model:
        try:
            return ollama
        except Exception:
            pass
    return openrouter if openrouter_api_key else openai_client

In [21]:
system_info = retrieve_system_info()
compile_command = ["clang", "-std=c17", "-O3", "-DNDEBUG", "main.c", "-o", "main"]
run_command = ["./main"]

In [36]:
SYSTEM_PROMPT = """
Convert Python code into high-performance C (C11/C17).
Respond only with C code. No explanation. No comments that mention conversion or AI.
Output must be identical to the Python program and run in the least time.
For integer arithmetic: use a type that can hold the full result (e.g. unsigned long long, or long long) so the numeric output matches Python exactly; avoid overflow.
"""

def user_prompt_for(python_code: str) -> str:
    try:
        si = system_info
    except NameError:
        si = retrieve_system_info()
    try:
        cc = compile_command
    except NameError:
        cc = ["clang", "-std=c17", "-O3", "-DNDEBUG", "main.c", "-o", "main"]
    return f"""
Port this Python code to C with the fastest implementation that produces identical output.
System information:
{si}
The C code will be written to main.c and compiled with:
{cc}
Respond only with C code.

```python
{python_code}
```
"""

In [37]:
def messages_for(python_code: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt_for(python_code)},
    ]

def strip_code_block(raw: str) -> str:
    s = (raw or "").strip()
    for prefix in ("```c", "```cpp", "```"):
        if s.startswith(prefix):
            s = s[len(prefix):].lstrip()
        if s.endswith("```"):
            s = s[:-3].rstrip()
    return s

def port(model: str, python_code: str) -> str:
    client = get_client(model)
    if not client:
        return "Error: No API client available for this model."
    api_model = model.split("/", 1)[1] if "/" in model and client is openai_client else model
    resp = client.chat.completions.create(
        model=api_model,
        messages=messages_for(python_code),
    )
    reply = (resp.choices[0].message.content or "").strip()
    return strip_code_block(reply)

In [38]:
def run_python(code: str) -> str:
    buf = io.StringIO()
    old = sys.stdout
    sys.stdout = buf
    try:
        exec(code, {"__builtins__": __builtins__})
        return buf.getvalue()
    except Exception as e:
        return f"Error: {e}"
    finally:
        sys.stdout = old

def write_and_compile_run(c_code: str, workdir: Path) -> str:
    main_c = workdir / "main.c"
    main_exe = workdir / "main"
    main_c.write_text(c_code, encoding="utf-8")
    try:
        subprocess.run(
            compile_command,
            cwd=workdir,
            check=True,
            capture_output=True,
            text=True,
        )
        r = subprocess.run(
            run_command,
            cwd=workdir,
            capture_output=True,
            text=True,
        )
        return r.stdout if r.returncode == 0 else (r.stderr or f"Exit {r.returncode}")
    except subprocess.CalledProcessError as e:
        return (e.stderr or e.stdout or str(e))

In [39]:
DEFAULT_PYTHON = """
import time

def sum_squares(n):
    total = 0
    for i in range(1, n + 1):
        total += i * i
    return total

n = 1_000_000
start = time.time()
result = sum_squares(n)
elapsed = time.time() - start
print(f"Sum of squares 1..{n} = {result}")
print(f"Time: {elapsed:.6f}s")
"""

In [40]:
workdir = Path.cwd() / "_c_build"
workdir.mkdir(exist_ok=True)

def convert(model: str, python_code: str) -> str:
    return port(model, python_code)

def run_c(c_code: str) -> str:
    if not c_code or not c_code.strip():
        return "(no C code to run)"
    return write_and_compile_run(c_code.strip(), workdir)

with gr.Blocks(title="Python to C Port") as ui:
    gr.Markdown("## Python → C port (Week 4 style)")
    with gr.Row():
        python_in = gr.Code(label="Python", value=DEFAULT_PYTHON, language="python", lines=18)
        c_out = gr.Code(label="C (generated)", language="c", lines=18)
    with gr.Row():
        model_dd = gr.Dropdown(MODELS, label="Model", value=MODELS[0])
        convert_btn = gr.Button("Convert to C")
    with gr.Row():
        py_out = gr.Textbox(label="Python output", lines=4)
        c_run_out = gr.Textbox(label="C output", lines=4)
    with gr.Row():
        run_py_btn = gr.Button("Run Python")
        run_c_btn = gr.Button("Run C")

    convert_btn.click(fn=convert, inputs=[model_dd, python_in], outputs=[c_out])
    run_py_btn.click(fn=run_python, inputs=[python_in], outputs=[py_out])
    run_c_btn.click(fn=run_c, inputs=[c_out], outputs=[c_run_out])

ui.launch()

* Running on local URL:  http://127.0.0.1:7901
* To create a public link, set `share=True` in `launch()`.
